# 사진 2장으로 스테레오 거리 측정 (Colab)

왼쪽/오른쪽 카메라로 찍은 사진 2장을 올리면 **시차(disparity) 맵과 거리 맵**을 계산합니다.

**사용법**: 위에서부터 셀을 순서대로 실행하세요 (`Shift + Enter`).

| 단계 | 내용 |
|---|---|
| 1 | 설정값 입력 (카메라 간격, 화각) |
| 2 | 왼쪽 사진, 오른쪽 사진 업로드 (+ 선택: `stereo_calib.npz`) |
| 3~5 | 정렬 → 시차 계산 → 거리 계산 (자동) |
| 6 | 결과 보기 (마우스를 올리면 거리 표시) |
| 7 | 원하는 지점 거리 측정 |

**촬영 팁**
- 두 사진은 **같은 카메라 높이, 같은 방향**으로, 옆으로만 평행 이동해서 찍기 (예: 6cm 간격)
- **왼쪽 = 피사체를 바라볼 때 왼쪽 카메라**
- 움직이는 물체 X, 무늬 없는 벽 X, 밝은 곳 O

**정확도**
- `stereo_calib.npz` (PC에서 `python stereo.py calibrate`로 만든 파일)를 올리면 → **정확한 측정**
- 없으면 → 화각(FOV)으로 초점거리를 추정하는 **근사 측정** (오차 10~30% 가능)

In [ ]:
#@title 1. 설정 { display-mode: "form" }
BASELINE_MM = 50.0     #@param {type:"number"}
#@markdown ↑ 두 카메라(촬영 위치) 사이 거리(mm). 캘리브레이션 파일을 쓰면 무시됨
HFOV_DEG = 78.0        #@param {type:"number"}
#@markdown ↑ 카메라 가로 화각(도). ESP32-CAM(OV2640 기본 렌즈) ≈ 66, 스마트폰 기본 카메라 ≈ 65~75
RESIZE_WIDTH = 640     #@param {type:"integer"}
#@markdown ↑ 계산용 가로 해상도. 큰 사진은 줄여야 빠르고 잘 맞음 (0 = 원본 유지)
RECTIFY_MODE = "auto"  #@param ["auto", "y-align", "none"]
#@markdown ↑ auto: 캘리브레이션 파일이 있으면 사용, 없으면 y-align (특징점으로 세로 어긋남 보정)
BLOCK_SIZE = 11         #@param {type:"slider", min:3, max:15, step:2}
MAX_DIST_CM = 500      #@param {type:"integer"}
#@markdown ↑ 이보다 먼 값은 표시하지 않음

In [ ]:
#@title 2. 사진 업로드
import os, subprocess, cv2, numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from google.colab import files

# 그래프 한글 깨짐 방지 (나눔고딕 설치)
FONT = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(FONT):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(FONT):
    font_manager.fontManager.addfont(FONT)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

def upload_one(msg):
    print(msg)
    up = files.upload()
    if not up:
        raise SystemExit("업로드가 취소되었습니다. 이 셀을 다시 실행하세요.")
    return list(up.keys())[0]

LEFT_PATH = upload_one("① 왼쪽 사진을 선택하세요")
RIGHT_PATH = upload_one("② 오른쪽 사진을 선택하세요")
print("③ (선택) stereo_calib.npz 를 올리려면 선택, 없으면 '취소'를 누르세요")
up = files.upload()
CALIB_PATH = next((k for k in up if k.endswith(".npz")), None)

print(f"\n왼쪽: {LEFT_PATH}\n오른쪽: {RIGHT_PATH}\n캘리브레이션: {CALIB_PATH or '없음 (근사 측정)'}")

In [ ]:
#@title 3. 사진 불러오기 + 정렬(평행화)
def load(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)   # 스마트폰 사진의 회전(EXIF)도 반영됨
    if img is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {path} (jpg/png만 지원)")
    return img

L0, R0 = load(LEFT_PATH), load(RIGHT_PATH)
if L0.shape != R0.shape:
    print(f"⚠ 두 사진 크기가 다릅니다 {L0.shape[:2]} vs {R0.shape[:2]} → 오른쪽을 왼쪽 크기로 맞춤")
    R0 = cv2.resize(R0, (L0.shape[1], L0.shape[0]))

# --- 캘리브레이션 파일 확인 ---
calib = None
if RECTIFY_MODE == "auto" and CALIB_PATH:
    calib = np.load(CALIB_PATH)
    cw, ch = (int(v) for v in calib["size"])
    if (L0.shape[1], L0.shape[0]) != (cw, ch):
        print(f"⚠ 사진 크기 {L0.shape[1]}x{L0.shape[0]} ≠ 캘리브레이션 크기 {cw}x{ch} → 캘리브레이션 사용 불가, y-align으로 진행")
        calib = None
mode = "calib" if calib is not None else ("none" if RECTIFY_MODE == "none" else "y-align")

def match_points(a, b, max_pts=4000):
    # SIFT 특징점 매칭 (Lowe ratio test)
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=max_pts)
    ka, da = sift.detectAndCompute(ga, None)
    kb, db = sift.detectAndCompute(gb, None)
    if da is None or db is None or len(ka) < 10 or len(kb) < 10:
        return np.empty((0, 2)), np.empty((0, 2))
    knn = cv2.BFMatcher().knnMatch(da, db, k=2)
    good = [m for m, n in (p for p in knn if len(p) == 2) if m.distance < 0.75 * n.distance]
    pa = np.float32([ka[m.queryIdx].pt for m in good])
    pb = np.float32([kb[m.trainIdx].pt for m in good])
    return pa, pb

if mode == "calib":
    size = (cw, ch)
    m1 = cv2.initUndistortRectifyMap(calib["K1"], calib["D1"], calib["R1"], calib["P1"], size, cv2.CV_32FC1)
    m2 = cv2.initUndistortRectifyMap(calib["K2"], calib["D2"], calib["R2"], calib["P2"], size, cv2.CV_32FC1)
    L = cv2.remap(L0, *m1, cv2.INTER_LINEAR)
    R = cv2.remap(R0, *m2, cv2.INTER_LINEAR)
    Q = calib["Q"]
    print("✔ 캘리브레이션 파일로 왜곡 보정 + 평행화 완료 (정확 측정 모드)")
else:
    # 계산 해상도로 축소
    if RESIZE_WIDTH and L0.shape[1] > RESIZE_WIDTH:
        s = RESIZE_WIDTH / L0.shape[1]
        L = cv2.resize(L0, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        R = cv2.resize(R0, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    else:
        L, R = L0.copy(), R0.copy()
    h, w = L.shape[:2]

    if mode == "y-align":
        # 오른쪽 사진의 '세로 위치'만 보정 (y' = y + a + b*x + c*y)
        # 가로(x)는 건드리지 않으므로 '무한히 먼 곳 = 시차 0' 관계가 유지됨
        pl, pr = match_points(L, R)
        if len(pl) < 20:
            print(f"⚠ 매칭점이 {len(pl)}개뿐이라 세로 보정을 건너뜁니다 (무늬가 많은 장면으로 다시 찍어 보세요)")
        else:
            dy = pl[:, 1] - pr[:, 1]
            A = np.c_[np.ones(len(pr)), pr[:, 0], pr[:, 1]]
            keep = np.abs(dy - np.median(dy)) < 3 * (np.median(np.abs(dy - np.median(dy))) + 1)
            for _ in range(5):   # 이상치 제거하며 반복 최소제곱
                coef, *_ = np.linalg.lstsq(A[keep], dy[keep], rcond=None)
                res = dy - A @ coef
                keep = np.abs(res) < max(2.0, 3 * np.median(np.abs(res[keep])))
            xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
            # yL = yR + a + b*x + c*yR  →  yR = (yL - a - b*x) / (1 + c)
            map_y = ((ys - coef[0] - coef[1] * xs) / (1 + coef[2])).astype(np.float32)
            R = cv2.remap(R, xs, map_y, cv2.INTER_LINEAR)
            print(f"✔ 세로 정렬 완료: 매칭점 {keep.sum()}/{len(pl)}개 사용, "
                  f"보정 전 세로 어긋남 중앙값 {np.median(np.abs(dy)):.1f}px → 보정 후 {np.median(np.abs(res[keep])):.2f}px")
    else:
        print("정렬 없이 진행 (두 사진이 이미 평행하다고 가정)")

    f_px = (w / 2) / np.tan(np.radians(HFOV_DEG) / 2)
    # Z = f*B/d 를 reprojectImageTo3D 형식의 Q 행렬로 구성
    Q = np.float32([[1, 0, 0, -w / 2],
                    [0, 1, 0, -h / 2],
                    [0, 0, 0, f_px],
                    [0, 0, 1 / BASELINE_MM, 0]])
    print(f"근사 측정 모드: 초점거리 ≈ {f_px:.0f}px (화각 {HFOV_DEG}°), 카메라 간격 {BASELINE_MM}mm")

# 정렬 확인: 같은 물체가 같은 가로줄 위에 있으면 OK
pair = np.hstack([L, R])
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(cv2.cvtColor(pair, cv2.COLOR_BGR2RGB))
for y in np.linspace(0, L.shape[0], 12)[1:-1]:
    ax.axhline(y, color="lime", lw=0.7)
ax.set_title("정렬 확인: 같은 물체가 초록 가로줄 기준으로 같은 높이에 있어야 함   (왼쪽 | 오른쪽)")
ax.axis("off"); plt.show()

In [ ]:
#@title 4. 시차(disparity) 계산
gl = cv2.cvtColor(L, cv2.COLOR_BGR2GRAY)
gr = cv2.cvtColor(R, cv2.COLOR_BGR2GRAY)
h, w = gl.shape

# 매칭점으로 실제 시차 범위를 측정해서 SGBM 탐색 범위를 자동 설정
pl, pr = match_points(L, R)
d_pts = pl[:, 0] - pr[:, 0] if len(pl) else np.array([])
d_pts = d_pts[np.abs(pl[:, 1] - pr[:, 1]) < 2] if len(pl) else d_pts
if len(d_pts) >= 10:
    if np.median(d_pts) < 0:
        print("⚠ 시차가 음수입니다 → 왼쪽/오른쪽 사진이 바뀐 것 같아 자동으로 교체합니다")
        L, R, gl, gr = R, L, gr, gl
        d_pts = -d_pts
    lo, hi = np.percentile(d_pts, [1, 99.5])
    min_disp = int(max(0, np.floor(lo) - 8))
    # 매칭점이 없는 더 가까운 물체도 잡도록 위쪽은 넉넉하게 (x1.5 + 16px)
    num_disp = int(np.ceil((hi * 1.5 + 16 - min_disp) / 16) * 16)
else:
    print("⚠ 매칭점이 부족해 기본 탐색 범위를 사용합니다")
    min_disp, num_disp = 0, 128
num_disp = int(min(max(num_disp, 16), (w - min_disp) // 16 * 16))
print(f"시차 탐색 범위: {min_disp} ~ {min_disp + num_disp} px")

bs = BLOCK_SIZE
sgbm = cv2.StereoSGBM_create(
    minDisparity=min_disp, numDisparities=num_disp, blockSize=bs,
    P1=8 * 3 * bs ** 2, P2=32 * 3 * bs ** 2, disp12MaxDiff=1,
    uniquenessRatio=10, speckleWindowSize=100, speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
raw = sgbm.compute(gl, gr)

# opencv-contrib가 있고 필요한 WLS 필터 함수가 모두 존재하는지 확인
if hasattr(cv2, "ximgproc") and hasattr(cv2.ximgproc, "createRightMatcher"):
    try:
        right_m = cv2.ximgproc.createRightMatcher(sgbm)
        wls = cv2.ximgproc.createDisparityWLSFilter(sgbm)
        wls.setLambda(8000); wls.setSigmaColor(1.2)
        raw = wls.filter(raw, gl, disparity_map_right=right_m.compute(gr, gl))
        print("✔ WLS 필터 적용")
    except Exception as e:
        print(f"⚠ WLS 필터 적용 실패: {e} (기본 시차 맵을 사용합니다)")
else:
    print("ℹ WLS 필터를 지원하지 않는 OpenCV 버전입니다 (기본 시차 맵을 사용합니다)")

disp = raw.astype(np.float32) / 16.0
valid_disp = disp > max(min_disp, 0.5)
print(f"유효 픽셀 비율: {valid_disp.mean() * 100:.0f}%")

In [ ]:
#@title 5. 거리 계산
pts3d = cv2.reprojectImageTo3D(disp, Q)
Z_cm = pts3d[:, :, 2] / 10.0                      # mm → cm
valid = valid_disp & np.isfinite(Z_cm) & (Z_cm > 0) & (Z_cm < MAX_DIST_CM)
depth_cm = np.where(valid, Z_cm, np.nan)

def distance_at(x, y, roi=7):
    # (x, y) 주변 roi x roi 영역의 유효 거리 중앙값 (cm)
    r = roi // 2
    patch = depth_cm[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1]
    vals = patch[np.isfinite(patch)]
    return float(np.median(vals)) if vals.size >= max(1, patch.size // 4) else None

if not valid.any():
    raise SystemExit("유효한 거리값이 없습니다. 3번 정렬 그림과 설정값(BASELINE_MM, HFOV_DEG)을 확인하세요.")
d_c = distance_at(w // 2, h // 2, roi=21)
print(f"측정된 거리 범위: {np.nanmin(depth_cm):.1f} ~ {np.nanmax(depth_cm):.1f} cm")
print(f"화면 중앙 거리: {d_c:.1f} cm" if d_c else "화면 중앙: 측정 불가")

In [ ]:
#@title 6. 결과 보기
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(cv2.cvtColor(L, cv2.COLOR_BGR2RGB)); axs[0].set_title("왼쪽 (정렬 후)")
dv = np.where(valid_disp, disp, np.nan)
im1 = axs[1].imshow(dv, cmap="jet"); axs[1].set_title("시차 (px) — 빨강 = 가까움")
plt.colorbar(im1, ax=axs[1], fraction=0.046)
im2 = axs[2].imshow(depth_cm, cmap="turbo_r"); axs[2].set_title("거리 (cm) — 빨강 = 가까움")
plt.colorbar(im2, ax=axs[2], fraction=0.046)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

# 마우스를 올리면 거리(cm)가 표시되는 인터랙티브 그림
import plotly.graph_objects as go
step = max(1, w // 400)                            # 너무 크면 브라우저가 느려지므로 축소
rgb = cv2.cvtColor(L, cv2.COLOR_BGR2RGB)[::step, ::step]
dsub = depth_cm[::step, ::step]
fig = go.Figure()
fig.add_trace(go.Image(z=rgb, x0=0, y0=0, dx=step, dy=step, hoverinfo="skip"))
fig.add_trace(go.Heatmap(z=dsub, x=np.arange(0, w, step), y=np.arange(0, h, step),
                         colorscale="Turbo", reversescale=True, opacity=0.35,
                         colorbar=dict(title="cm"),
                         hovertemplate="x=%{x}, y=%{y}<br>거리: %{z:.1f} cm<extra></extra>"))
fig.update_layout(title="마우스를 올리면 거리 표시 (색이 없는 곳 = 측정 불가)",
                  width=900, height=int(900 * h / w) + 60, margin=dict(l=0, r=0, t=40, b=0))
fig.update_yaxes(autorange="reversed", scaleanchor="x", showticklabels=False)
fig.update_xaxes(showticklabels=False)
fig.show()

In [ ]:
#@title 7. 원하는 지점 거리 측정 { run: "auto" }
#@markdown 위 그림에서 마우스로 확인한 x, y 좌표를 입력하세요
POINT_X = 258  #@param {type:"integer"}
POINT_Y = 257  #@param {type:"integer"}
ROI = 31        #@param {type:"slider", min:1, max:31, step:2}

x, y = int(np.clip(POINT_X, 0, w - 1)), int(np.clip(POINT_Y, 0, h - 1))
d = distance_at(x, y, ROI)
view = cv2.cvtColor(L, cv2.COLOR_BGR2RGB).copy()
cv2.drawMarker(view, (x, y), (0, 255, 0), cv2.MARKER_CROSS, 30, 2)
plt.figure(figsize=(9, 7)); plt.imshow(view); plt.axis("off")
plt.title(f"({x}, {y}) 거리: {d:.1f} cm" if d else f"({x}, {y}) 측정 불가 — 무늬가 있는 곳을 골라 보세요")
plt.show()

In [ ]:
#@title 8. (선택) 결과 다운로드
np.save("depth_cm.npy", depth_cm)
vis = cv2.applyColorMap(cv2.normalize(np.nan_to_num(-depth_cm, nan=np.nanmin(-depth_cm)), None, 0, 255,
                                      cv2.NORM_MINMAX).astype(np.uint8), cv2.COLORMAP_TURBO)
cv2.imwrite("depth_color.png", vis)
files.download("depth_cm.npy"); files.download("depth_color.png")

## 결과가 이상할 때

| 증상 | 해결 |
|---|---|
| 3번 그림에서 같은 물체 높이가 다름 | 카메라를 같은 높이·방향으로 다시 촬영, 또는 캘리브레이션 파일 사용 |
| 시차 맵이 대부분 비어 있음 | 무늬가 많은 장면, 밝은 곳에서 촬영. `BLOCK_SIZE` 7~9로 키우기 |
| 거리가 일정 비율로 틀림 | `HFOV_DEG`, `BASELINE_MM` 확인. 실제 거리를 아는 물체로 화각 값을 조정 |
| 가까운 물체가 측정 안 됨 | 너무 가까움 (간격 6cm면 약 30cm 이상 권장) |
| 좌우가 바뀜 | 4번 셀이 자동으로 감지해 교체함 |